# 1. Limpeza dos dados: Ames Housing

**Objetivo:** ler o CSV bruto (`data/raw/`), corrigir os problemas e salvar uma versão limpa em `data/processed/`.

Decisões tomadas aqui:
- remover colunas de identificação (índice, `Order`, `PID`);
- padronizar nomes das colunas (snake_case);
- tratar valores ausentes que significam **"não possui"** (ex.: sem garagem, sem piscina);
- corrigir um ano impossível em `Garage Yr Blt`;
- remover 5 outliers de área (recomendação do autor do dataset);
- manter os demais NaN reais, que o LightGBM aceita nativamente.

In [2]:
from pathlib import Path
import pandas as pd

RAW = Path("../data/raw/ames_housing.csv")
OUT = Path("../data/processed/ames_clean.parquet")

## Leitura

Usamos `keep_default_na=False` e `na_values=[""]` porque, neste CSV, o texto `"None"` é uma **categoria válida** (ex.: `Mas Vnr Type = None`, sem revestimento) e não deve virar NaN. Só as células vazias viram NaN.

In [3]:
df = pd.read_csv(RAW, keep_default_na=False, na_values=[""])
print(df.shape)
df.head(3)

(2930, 83)


,Unnamed: 0,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000


## Colunas de identificação e nomes

In [4]:
# Remove índices soltos
df = df.drop(columns=["Unnamed: 0", "Order", "PID"])

# Nomes: minúsculo, sem espaços/barras
def limpar_nome(c):
    return c.strip().lower().replace("/", "_").replace(" ", "_")

df.columns = [limpar_nome(c) for c in df.columns]
print(df.columns.tolist())

['ms_subclass', 'ms_zoning', 'lot_frontage', 'lot_area', 'street', 'alley', 'lot_shape', 'land_contour', 'utilities', 'lot_config', 'land_slope', 'neighborhood', 'condition_1', 'condition_2', 'bldg_type', 'house_style', 'overall_qual', 'overall_cond', 'year_built', 'year_remod_add', 'roof_style', 'roof_matl', 'exterior_1st', 'exterior_2nd', 'mas_vnr_type', 'mas_vnr_area', 'exter_qual', 'exter_cond', 'foundation', 'bsmt_qual', 'bsmt_cond', 'bsmt_exposure', 'bsmtfin_type_1', 'bsmtfin_sf_1', 'bsmtfin_type_2', 'bsmtfin_sf_2', 'bsmt_unf_sf', 'total_bsmt_sf', 'heating', 'heating_qc', 'central_air', 'electrical', '1st_flr_sf', '2nd_flr_sf', 'low_qual_fin_sf', 'gr_liv_area', 'bsmt_full_bath', 'bsmt_half_bath', 'full_bath', 'half_bath', 'bedroom_abvgr', 'kitchen_abvgr', 'kitchen_qual', 'totrms_abvgrd', 'functional', 'fireplaces', 'fireplace_qu', 'garage_type', 'garage_yr_blt', 'garage_finish', 'garage_cars', 'garage_area', 'garage_qual', 'garage_cond', 'paved_drive', 'wood_deck_sf', 'open_porch

## Valores ausentes

Antes de tratar, vale ver onde estão. Em muitas colunas categóricas, o NaN significa **ausência do item** (casa sem garagem, sem piscina etc.), e não dado faltando.

In [5]:
faltando = df.isna().sum()
faltando = faltando[faltando > 0].sort_values(ascending=False)
pd.DataFrame({"n": faltando, "%": (faltando / len(df) * 100).round(1)})

,n,%
pool_qc,2917,99.6
misc_feature,2824,96.4
alley,2732,93.2
fence,2358,80.5
fireplace_qu,1422,48.5
lot_frontage,490,16.7
garage_finish,159,5.4
garage_qual,159,5.4
garage_cond,159,5.4
garage_yr_blt,159,5.4


In [6]:
# NaN = "não possui"
sem_item = [
    "alley", "bsmt_qual", "bsmt_cond", "bsmt_exposure", "bsmtfin_type_1", "bsmtfin_type_2",
    "fireplace_qu", "garage_type", "garage_finish", "garage_qual", "garage_cond",
    "pool_qc", "fence", "misc_feature",
]
df[sem_item] = df[sem_item].fillna("None")

## Correções pontuais

- `ms_subclass` é um **código** de tipo de imóvel (20, 60, 120...), não uma quantidade. Vira texto.
- `garage_yr_blt` tem um ano impossível (maior que 2010, o último ano de venda).

In [7]:
df["ms_subclass"] = df["ms_subclass"].astype(str)

print("garage_yr_blt máx antes:", df["garage_yr_blt"].max())
df.loc[df["garage_yr_blt"] > df["yr_sold"], "garage_yr_blt"] = pd.NA
print("garage_yr_blt máx depois:", df["garage_yr_blt"].max())

garage_yr_blt máx antes: 2207.0
garage_yr_blt máx depois: 2010.0


## Outliers de área

O autor do dataset (De Cock) recomenda remover casas com `gr_liv_area` acima de 4000 pés², pois há vendas atípicas (inclusive casas enormes vendidas por preço baixo). São só 5 linhas.

In [8]:
out = df[df["gr_liv_area"] > 4000][["gr_liv_area", "saleprice", "yr_sold"]]
display(out)
df = df[df["gr_liv_area"] <= 4000].reset_index(drop=True)
print("linhas após remover:", len(df))

,gr_liv_area,saleprice,yr_sold
1498,5642,160000,2008
1760,4476,745000,2007
1767,4316,755000,2007
2180,5095,183850,2007
2181,4676,184750,2007


linhas após remover: 2925


## Validações

In [9]:
assert df["saleprice"].gt(0).all(), "preço <= 0"
assert df["gr_liv_area"].gt(0).all(), "área <= 0"
assert df.duplicated().sum() == 0, "linhas duplicadas"
assert df["yr_sold"].between(2006, 2010).all()

print("OK. shape final:", df.shape)
print("\nVendas por ano:")
print(df["yr_sold"].value_counts().sort_index())

OK. shape final: (2925, 80)

Vendas por ano:
yr_sold
2006    625
2007    690
2008    621
2009    648
2010    341
Name: count, dtype: int64


## Salvar

In [10]:
df.to_parquet(OUT, index=False)
pd.read_parquet(OUT).shape

(2925, 80)